In [1]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")


   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
              ✗ CPLEX
  0 dependencies successfully precompiled in 3 seconds. 59 already precompiled.

The following 1 direct dependency failed to precompile:

CPLEX 

Failed to precompile CPLEX [a076750e-1247-5638-91d2-ce28b192dca0] to "/home/husted42/.julia/compiled/v1.12/CPLEX/jl_H16QgS".
ERROR: LoadError: CPLEX not properly installed. Please run Pkg.build("CPLEX")
Stacktrace:
  [1] error(s::String)
    @ Base ./error.jl:44
  [2] top-level scope
    @ ~/.julia/packages/CPLEX/5jmjD/src/CPLEX.jl:12
  [3] include(mod::Module, _path::String)
    @ Base ./Base.jl:306
  [4] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{String}, dl_load_path::Vector{String}, load_path::Vector{String}, concrete_deps::Vector{Pair{Base.P

# Blending 2

In [6]:
# agiso@dtu.dk
using JuMP, HiGHS

##### ----- variable ----- #####
M = 6
I = 5

months = 1:M
indencies = 1:I

# Make a matrix with cost
cost = [
    110 120 130 110 115;
    130 130 110 90 115;
    110 140 130 100 95;
    120 110 120 120 125;
    100 120 150 110 105;
    90 100 140 80 135
]

earnings = 150
storage_cost = 5

# Production limit
prod_limit_1 = 200
prod_limit_2 = 250

# Hardness
hardeness = [8.8, 6.1, 2.0, 4.2, 5.0]
h_lower = 3
h_upper = 6

# storage
s_start = 500
s_end = 500
s_high = 1000

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, p[months, indencies] >= 0)
@variable(model, s[months, indencies] >= 0)
@variable(model, sale[months, indencies] >= 0)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(earnings * sale[m, i] for m in months, i in indencies) -
    sum(cost[m, i] * p[m, i] for m in months, i in indencies) -
    sum(storage_cost * s[m, i] for m in months, i in indencies)
)

##### ----- Simple constraints ----- #####
# Production capacity constraints
@constraint(model, [m in months],
    sale[m, 1] + sale[m, 2] <= prod_limit_1
)
@constraint(model, [m in months],
    sale[m, 3] + sale[m, 4] + sale[m, 5] <= prod_limit_2
)

# Final storage constraints
@constraint(model, [i in indencies],
    s[6, i] == s_end
)
# Storage capacity constraints
@constraint(model, [m in months, i in indencies],
    s[m, i] <= s_high
)

#### ----- Constraints ----- #####
# We store what we do not sell + storage from last month
@constraint(model, [m in months, i in indencies],
    s[m, i] == (m > 1 ? s[m-1, i] : s_start) + p[m, i] - sale[m, i]
)

# Hardness constraints
@constraint(model, [m in months],
    sum(hardeness[i] * sale[m,i] for i in indencies) >= h_lower * sum(sale[m,i] for i in indencies)
)
@constraint(model, [m in months],
    sum(hardeness[i] * sale[m,i] for i in indencies) <= h_upper * sum(sale[m,i] for i in indencies)
)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\nProduction:")
println(value.(p))

println("\nSales:")
println(value.(sale))

println("\nStorage:")
println(value.(s))


Optimal solution:
z = 107842.59259259255

Production:
2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:6
    Dimension 2, 1:5
And data, a 6×5 Matrix{Float64}:
   0.0                0.0              0.0    0.0    0.0
   0.0                0.0              0.0    0.0    0.0
   0.0                0.0              0.0    0.0  250.0
   0.0                0.0              0.0    0.0    0.0
   0.0                0.0              0.0    0.0  500.0
 659.2592592592592  540.7407407407408  0.0  750.0    0.0

Sales:
2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:6
    Dimension 2, 1:5
And data, a 6×5 Matrix{Float64}:
  85.18518518518518   114.81481481481482   0.0    0.0  250.0
 159.2592592592592     40.74074074074079   0.0  250.0    0.0
  11.111111111111228  188.88888888888877   0.0    0.0  250.0
  85.18518518518518   114.81481481481482   0.0   -0.0  250.0
 159.2592592592592     40.74074074074079  -0.0  250.0    0.0
 159.2592592592592 

#